# 대출 상환 계산기 (원리금균등상환)

주택담보대출 원리금균등상환 방식의 월 상환액을 계산하고, 상환 스케줄을 시각화한다.

1. 대출 파라미터 설정 (원금, 금리, 기간)
2. 월 상환액 계산 (원리금균등상환 공식)
3. 상환 스케줄 테이블 생성
4. 원금 vs 이자 상환 추이 시각화
5. 총 이자 납부액 분석

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline

In [ ]:
# 대출 파라미터 설정
# 필요에 따라 값을 변경하여 사용한다.

PRINCIPAL = 300_000_000   # 대출 원금 (원) - 3억
ANNUAL_RATE = 3.5         # 연이율 (%)
LOAN_YEARS = 30           # 대출 기간 (년)

print(f"대출 원금: {PRINCIPAL:,.0f}원 ({PRINCIPAL // 10000:,.0f}만원)")
print(f"연이율: {ANNUAL_RATE}%")
print(f"대출 기간: {LOAN_YEARS}년 ({LOAN_YEARS * 12}개월)")

In [ ]:
# 원리금균등상환 월 상환액 계산
# 공식: M = P * r * (1+r)^n / ((1+r)^n - 1)
# P: 원금, r: 월이율, n: 총 상환 개월 수

def calculate_monthly_payment(principal, annual_rate, years):
    """원리금균등상환 월 상환액을 계산한다."""
    monthly_rate = annual_rate / 100 / 12
    n_months = years * 12
    
    if monthly_rate == 0:
        return principal / n_months
    
    payment = principal * monthly_rate * (1 + monthly_rate)**n_months / (
        (1 + monthly_rate)**n_months - 1
    )
    return payment

monthly_payment = calculate_monthly_payment(PRINCIPAL, ANNUAL_RATE, LOAN_YEARS)

print(f"월 상환액: {monthly_payment:,.0f}원 ({monthly_payment / 10000:,.1f}만원)")
print(f"총 상환액: {monthly_payment * LOAN_YEARS * 12:,.0f}원")
print(f"총 이자: {monthly_payment * LOAN_YEARS * 12 - PRINCIPAL:,.0f}원")

In [ ]:
# 상환 스케줄 테이블 생성
def generate_amortization_schedule(principal, annual_rate, years):
    """원리금균등상환 상환 스케줄을 생성한다."""
    monthly_rate = annual_rate / 100 / 12
    n_months = years * 12
    monthly_payment = calculate_monthly_payment(principal, annual_rate, years)
    
    schedule = []
    balance = principal
    
    for month in range(1, n_months + 1):
        interest = balance * monthly_rate
        principal_paid = monthly_payment - interest
        balance -= principal_paid
        
        # 마지막 달 잔액 보정
        if month == n_months:
            balance = 0
        
        schedule.append({
            '회차': month,
            '월상환액': round(monthly_payment),
            '원금상환': round(principal_paid),
            '이자상환': round(interest),
            '잔액': round(max(balance, 0)),
        })
    
    return pd.DataFrame(schedule)

schedule_df = generate_amortization_schedule(PRINCIPAL, ANNUAL_RATE, LOAN_YEARS)

print(f"상환 스케줄 (총 {len(schedule_df)}개월)")
print("\n--- 처음 12개월 ---")
display(schedule_df.head(12))
print("\n--- 마지막 12개월 ---")
display(schedule_df.tail(12))

In [ ]:
# 원금 vs 이자 상환 추이 (영역 차트)
fig, ax = plt.subplots(figsize=(14, 6))

months = schedule_df['회차']
ax.fill_between(months, 0, schedule_df['원금상환'], alpha=0.7, label='원금 상환', color='#3498db')
ax.fill_between(months, schedule_df['원금상환'],
                schedule_df['원금상환'] + schedule_df['이자상환'],
                alpha=0.7, label='이자 상환', color='#e74c3c')

ax.set_title('월별 원금 vs 이자 상환 추이 (원리금균등상환)', fontsize=14, fontweight='bold')
ax.set_xlabel('상환 회차 (개월)')
ax.set_ylabel('상환액 (원)')
ax.legend(loc='center right')

# 상환 기간의 주요 포인트 표시
half_point = len(schedule_df) // 2
ax.axvline(x=half_point, color='gray', linestyle=':', alpha=0.5)
ax.text(half_point + 2, ax.get_ylim()[1] * 0.9, f'{half_point}개월\n(절반 지점)',
        fontsize=9, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# 잔액 감소 추이
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(schedule_df['회차'], schedule_df['잔액'] / 10000, color='#2c3e50', linewidth=2)
ax.fill_between(schedule_df['회차'], schedule_df['잔액'] / 10000, alpha=0.15, color='#2c3e50')

ax.set_title('대출 잔액 감소 추이', fontsize=14, fontweight='bold')
ax.set_xlabel('상환 회차 (개월)')
ax.set_ylabel('대출 잔액 (만원)')

# 주요 시점 표시
for pct in [0.25, 0.5, 0.75]:
    target = PRINCIPAL * (1 - pct)
    idx = (schedule_df['잔액'] - target).abs().idxmin()
    row = schedule_df.iloc[idx]
    ax.annotate(
        f"{int(pct*100)}% 상환\n({int(row['회차'])}개월)",
        xy=(row['회차'], row['잔액'] / 10000),
        xytext=(row['회차'] + 15, row['잔액'] / 10000 + PRINCIPAL / 10000 * 0.05),
        fontsize=9,
        arrowprops=dict(arrowstyle='->', color='gray'),
        color='#e74c3c'
    )

plt.tight_layout()
plt.show()

In [ ]:
# 누적 원금 vs 누적 이자 비교
schedule_df['누적원금'] = schedule_df['원금상환'].cumsum()
schedule_df['누적이자'] = schedule_df['이자상환'].cumsum()

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(schedule_df['회차'], schedule_df['누적원금'] / 10000,
        label='누적 원금 상환', color='#3498db', linewidth=2)
ax.plot(schedule_df['회차'], schedule_df['누적이자'] / 10000,
        label='누적 이자 납부', color='#e74c3c', linewidth=2)

# 교차점 찾기 (원금 > 이자 되는 시점)
cross_idx = (schedule_df['누적원금'] > schedule_df['누적이자']).idxmax()
cross_month = schedule_df.iloc[cross_idx]['회차']
ax.axvline(x=cross_month, color='gray', linestyle='--', alpha=0.5)
ax.text(cross_month + 3, ax.get_ylim()[1] * 0.5,
        f'{int(cross_month)}개월\n(누적 원금 > 이자)',
        fontsize=9, color='gray')

ax.set_title('누적 원금 상환 vs 누적 이자 납부', fontsize=14, fontweight='bold')
ax.set_xlabel('상환 회차 (개월)')
ax.set_ylabel('누적 금액 (만원)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 총 상환 요약 (파이 차트)
total_interest = schedule_df['이자상환'].sum()
total_paid = PRINCIPAL + total_interest

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 파이 차트: 원금 vs 이자
labels = ['원금', '이자']
sizes = [PRINCIPAL, total_interest]
colors_pie = ['#3498db', '#e74c3c']
explode = (0, 0.05)

axes[0].pie(sizes, explode=explode, labels=labels, colors=colors_pie,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[0].set_title('총 상환액 구성', fontsize=13, fontweight='bold')

# 요약 텍스트
summary_text = (
    f"대출 원금: {PRINCIPAL:>15,.0f}원\n"
    f"총 이자: {total_interest:>15,.0f}원\n"
    f"총 상환액: {total_paid:>15,.0f}원\n"
    f"\n"
    f"월 상환액: {monthly_payment:>15,.0f}원\n"
    f"이자 비율: {total_interest / total_paid * 100:>14.1f}%\n"
    f"대출 기간: {LOAN_YEARS}년 ({LOAN_YEARS * 12}개월)"
)

axes[1].text(0.1, 0.5, summary_text, fontsize=12, fontfamily='monospace',
             verticalalignment='center', transform=axes[1].transAxes,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1].axis('off')
axes[1].set_title('상환 요약', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 금리 민감도 분석
# 다양한 금리에서의 월 상환액과 총 이자를 비교한다.
rates = np.arange(2.0, 7.5, 0.5)
sensitivity = []

for rate in rates:
    mp = calculate_monthly_payment(PRINCIPAL, rate, LOAN_YEARS)
    total = mp * LOAN_YEARS * 12
    interest = total - PRINCIPAL
    sensitivity.append({
        '연이율(%)': rate,
        '월상환액(만원)': round(mp / 10000, 1),
        '총이자(만원)': round(interest / 10000, 0),
        '총상환액(만원)': round(total / 10000, 0),
    })

sens_df = pd.DataFrame(sensitivity)

fig, ax1 = plt.subplots(figsize=(12, 5))

color1 = '#3498db'
color2 = '#e74c3c'

ax1.bar(sens_df['연이율(%)'].astype(str), sens_df['월상환액(만원)'],
        color=color1, alpha=0.7, label='월 상환액')
ax1.set_xlabel('연이율 (%)')
ax1.set_ylabel('월 상환액 (만원)', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(sens_df['연이율(%)'].astype(str), sens_df['총이자(만원)'],
         color=color2, marker='o', linewidth=2, label='총 이자')
ax2.set_ylabel('총 이자 (만원)', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

ax1.set_title(f'금리별 상환 비교 (원금 {PRINCIPAL//10000:,}만원, {LOAN_YEARS}년)',
              fontsize=14, fontweight='bold')

# 범례 합치기
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

print("\n금리 민감도 분석 테이블:")
sens_df

## 분석 요약

- **원리금균등상환** 방식은 매달 동일한 금액을 상환하며, 초기에는 이자 비중이 크고 점차 원금 비중이 커진다.
- 금리가 1%p 오를 때마다 총 이자 부담이 크게 증가한다.
- 대출 기간이 길수록 월 상환 부담은 줄지만 총 이자는 늘어나므로, 상환 여력에 맞는 기간 설정이 중요하다.
- 실제 대출 시에는 중도상환수수료, 변동금리 리스크 등도 함께 고려해야 한다.